# Cost-Sensitive Credit Risk Scorecard — a narrative walkthrough

This notebook narrates the project's findings by importing from the actual
modules in this repository (`data/`, `evaluation/`, `fairness/`,
`explainability/`) and reading the CSVs those modules already produced and
committed under `reports/`. It is not a place where numbers are re-derived
from scratch — the full pipeline (raw CSV to every report and figure) is
`python run_all.py`, which takes on the order of an hour and is not
re-run here. What this notebook *does* run live are a few small,
fast demonstrations of the actual mechanisms behind the headline
findings, so "imports from the modules" means something concrete rather
than copy-pasted numbers with an import statement bolted on.

Full detail, every table, and the reasoning behind each modelling choice:
[`README.md`](../README.md), [`DECISIONS.md`](../DECISIONS.md),
[`MODEL_CARD.md`](../MODEL_CARD.md).

In [1]:
import sys
from pathlib import Path

# Work from the repo root regardless of where Jupyter's cwd landed.
REPO_ROOT = Path.cwd() if (Path.cwd() / "README.md").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)
REPORTS = REPO_ROOT / "reports"
assert (REPORTS / "leakage_experiment.csv").exists(), "run from inside the repo (or notebooks/)"
print("repo root found, reports/ readable.")

repo root found, reports/ readable.


## 1. The leakage trap

Phase 1 classified all 151 raw columns individually — an allow-list, not a
drop-list — before any model was trained. Loading the classification
itself (not a copy of the printed counts) confirms the numbers reported in
the README: it is the same `FEATURE_AUDIT` dict `data/feature_audit.py`
exports, and `models/leakage_experiment.py` imports, to build the feature
sets below.

In [2]:
import data.feature_audit as fa

decisions = pd.Series([v[0] for v in fa.FEATURE_AUDIT.values()]).value_counts()
print(f"{len(fa.FEATURE_AUDIT)} raw columns classified:")
print(decisions.to_string())

print("\nA sample of the *reasons*, not just the label -- this is what makes it an audit:")
for col in ["recoveries", "last_fico_range_high", "grade", "zip_code"]:
    reason = fa.FEATURE_AUDIT[col]
    print(f"  {col:22s} {reason[0]:14s} {reason[1]}")

151 raw columns classified:
keep            101
drop_leakage     38
drop_other        8
review            4

A sample of the *reasons*, not just the label -- this is what makes it an audit:
  recoveries             drop_leakage   money recovered after charge-off -- literally post-outcome
  last_fico_range_high   drop_leakage   updated FICO from a post-origination pull -- will visibly crater for defaulters
  grade                  review         LC's own risk grade -- an output of their underwriting model, not an input to ours
  zip_code               keep           3-digit ZIP prefix at application -- also our geography proxy for the fairness audit


In [3]:
leakage = pd.read_csv(REPORTS / "leakage_experiment.csv")
leakage[["regime", "n_features", "pr_auc", "roc_auc", "brier"]]

,regime,n_features,pr_auc,roc_auc,brier
0,clean,97,0.194439,0.697149,0.083007
1,grade_kept,101,0.203679,0.709450,0.082428
2,naive,139,0.997205,0.999645,0.003437


`clean` (97 features, the disciplined pre-decision set) vs. whichever
regime in this table used every available column — the ~0.80 PR-AUC gap
is finding 1. A model scoring near 1.0 on this dataset is a leakage
symptom, not an achievement.

## 2. Temporal validation: the honest constraint costs more than the split type

Four regimes at matched sample sizes, evaluated per fold. `embargoed` is
the regime an actual underwriting deployment would face: train only on
loans whose 18-month outcome was already knowable at model-fit time.

In [4]:
tv = pd.read_csv(REPORTS / "temporal_validation.csv")

# random and random_matched/temporal/embargoed use different fold labels
# (random reruns on shuffled seeds "r1"-"r4"; the other three share the
# real cohort years 2014-2017) -- pivoting them together produces a table
# full of NaNs that looks broken rather than merely apples-to-oranges.
# README's own reasoning is why: random's test population differs fold to
# fold, so per the project's own standard it "should not be used for the
# gap" and is reported here separately, for reference only.
year_regimes = tv[tv.regime != "random"]
print("Year-keyed regimes (comparable, same test cohort per fold):")
display(year_regimes.pivot_table(index="fold", columns="regime", values="pr_auc"))

random_only = tv[tv.regime == "random"]
print(f"\nrandom (plain 80/20, mixed-cohort test pool, for reference only): "
      f"mean PR-AUC {random_only.pr_auc.mean():.4f}, sd {random_only.pr_auc.std():.4f}")

Year-keyed regimes (comparable, same test cohort per fold):


regime,embargoed,random_matched,temporal
fold,,,
2014,0.122973,0.161306,0.157332
2015,0.168328,0.200813,0.196532
2016,0.188322,0.203752,0.197982
2017,0.170940,0.178754,0.178058



random (plain 80/20, mixed-cohort test pool, for reference only): mean PR-AUC 0.1869, sd 0.0020


## 3. Does gradient boosting beat a 1960s-vintage scorecard?

Four models, same embargoed folds.

In [5]:
baselines = pd.read_csv(REPORTS / "baselines.csv")
baselines.groupby("model")[["pr_auc", "roc_auc", "brier"]].agg(["mean", "std"]).round(4)

pr_auc         roc_auc           brier        
                mean     std    mean     std    mean     std
model                                                       
lightgbm      0.1666  0.0287  0.6598  0.0273  0.0859  0.0057
logistic_raw  0.1634  0.0298  0.6577  0.0269  0.0864  0.0055
logistic_woe  0.1650  0.0259  0.6583  0.0215  0.0857  0.0063
trivial       0.0974  0.0086  0.5000  0.0000  0.0882  0.0072

LightGBM's mean edge over the WoE-logistic scorecard is inside its own
fold-to-fold noise (finding 7) — confirmed by the paired, tuned-vs-tuned
comparison in the README, which this table alone can't show since it
isn't paired here.

## 4. Imbalance handling: every treatment made ranking no better and calibration worse

`none` is the baseline every treatment below is measured against.

In [6]:
imb = pd.read_csv(REPORTS / "imbalance.csv")
imb.groupby("treatment")[["pr_auc", "mean_pred", "true_rate"]].mean().round(4)

,pr_auc,mean_pred,true_rate
treatment,,,
class_weight,0.1634,0.3759,0.0974
none,0.1666,0.0799,0.0974
smote,0.1651,0.0907,0.0974
smote_rounded,0.1404,0.1498,0.0974
undersample,0.1609,0.4633,0.0974


`mean_pred` for `class_weight` and `undersample` sit 4-5x `true_rate` —
rebalancing shifts every score up without improving what actually ranks
defaulters against non-defaulters (`pr_auc` barely moves). This is the
calibration damage finding 9 documents.

## 5. Calibration decays under drift, live demonstration of *why* it's measured this way

Every calibrator here is fit on one cohort and scored on a test cohort
two-plus years later — never scored on its own training data, which is
the only way to see the drift finding 11 reports. The `cal_*` columns are
in-cohort; the bare columns are the test-cohort score actually used
everywhere else in this project.

In [7]:
cal = pd.read_csv(REPORTS / "calibration.csv")
cal.groupby(["model", "calibrator"])[["cal_ece", "ece"]].mean().round(4).rename(
    columns={"cal_ece": "ECE on its own cohort", "ece": "ECE on the test cohort"})

ECE on its own cohort  ECE on the test cohort
model        calibrator                                               
lightgbm     isotonic                   0.0000                  0.0191
             none                       0.0147                  0.0308
             platt                      0.0039                  0.0190
logistic_woe isotonic                   0.0000                  0.0183
             none                       0.0080                  0.0203
             platt                      0.0029                  0.0183

Isotonic reaches near-zero ECE on the cohort it was fit to and lands in
the same place as Platt two years later. Flexibility buys nothing once
the population has moved — that's the point finding 11 makes with a
whole paragraph; here it's one `groupby`.

## 6. Cost-sensitive decisioning — the exact threshold search, live

`evaluation/decisioning.py::best_threshold` finds the profit-maximising
cutoff by an *exact* search over every distinct calibrated score, not a
grid. Demonstrated here on a small synthetic book with a hand-computable
answer, exactly like `tests/test_decisioning.py` checks it against brute
force: at `lgd_rate=0.65`, `margin_rate=0.16`, a block of loans scored at
p=0.10 is profitable to approve (0.16×0.90 − 0.65×0.10 = +$0.079/dollar)
and a block at p=0.30 is not (0.16×0.70 − 0.65×0.30 = −$0.083/dollar). The
search should approve the whole p=0.10 block and reject the whole p=0.30
block — since every safe loan shares the exact same score, the true
optimum sits *at* 0.10, not strictly above it (there's no data point
between 0.10 and 0.30 to distinguish "at" from "just above"), so the
correct answer is `0.10 <= threshold < 0.30`, which is exactly what
`tests/test_decisioning.py::test_threshold_separates_a_profitable_block_from_an_unprofitable_one`
asserts rather than the looser "strictly between" a hand-wave might
suggest.

In [8]:
from evaluation.decisioning import best_threshold, profit_at

rng = np.random.default_rng(0)
n_safe, n_risky = 2000, 2000
p_demo = np.concatenate([np.full(n_safe, 0.10), np.full(n_risky, 0.30)])
y_demo = (rng.uniform(0, 1, n_safe + n_risky) < p_demo).astype(int)
amt_demo = np.full(n_safe + n_risky, 5_000.0)

thr, profit = best_threshold(y_demo, p_demo, amt_demo, lgd_rate=0.65, margin_rate=0.16)
assert 0.10 <= thr < 0.30, thr
print(f"threshold found: {thr:.3f}  (0.10 <= threshold < 0.30, as expected)")
print(f"expected profit per applicant at that threshold: ${profit / len(p_demo):.2f}")

threshold found: 0.100  (0.10 <= threshold < 0.30, as expected)
expected profit per applicant at that threshold: $0.05


Now the real numbers, on the actual four folds, using Phase 5's own real unit economics:

In [9]:
thresholds = pd.read_csv(REPORTS / "decisioning_thresholds.csv")
thresholds[thresholds.model == "logistic_woe"][
    ["fold", "thr_realistic", "approve_rate_realistic", "profit_realistic_on_test", "vs_naive_gain_per_applicant"]
]

,fold,thr_realistic,approve_rate_realistic,profit_realistic_on_test,vs_naive_gain_per_applicant
0,2014,0.228354,0.99167,1379.438835,-6.972640
2,2015,0.199510,0.98140,1227.397072,6.795444
4,2016,0.236839,0.99388,1066.127286,7.402259
6,2017,0.196839,0.98091,1217.086398,14.446953


## 7. Fairness audit: the impossibility result, made numeric

Every proxy shows both demographic-parity and equal-opportunity gaps
nonzero at once, at Phase 5's own shared cost-optimal threshold — never a
threshold picked to make this section look better or worse.

In [10]:
gaps = pd.read_csv(REPORTS / "fairness_gaps.csv")
baseline_gaps = gaps[gaps.scenario == "baseline"]
baseline_gaps.groupby(["proxy", "model"])[["dp_gap", "disparate_impact_ratio", "eo_tpr_gap"]].mean().round(4)

dp_gap  disparate_impact_ratio  eo_tpr_gap
proxy           model                                                   
emp_length      lightgbm      0.0194                  0.9803      0.0191
                logistic_woe  0.0554                  0.9443      0.0515
geo_race_proxy  lightgbm      0.0096                  0.9902      0.0112
                logistic_woe  0.0113                  0.9886      0.0102
income_quintile lightgbm      0.0183                  0.9814      0.0167
                logistic_woe  0.0240                  0.9760      0.0221

`emp_length` has the largest gaps of the three proxies on both criteria at
once — consistent with it having the largest spread in group base rates
(see `fairness/audit.py` and DECISIONS.md decision 12).

## 8. Explainability — the exact linear SHAP closed form, live

For an additively separable model (any linear model), the exact Shapley
value of feature *i* is `coef_i * (x_i - E[x_i])` — no approximation, no
independence assumption, because there are no interaction terms to
apportion credit for. Demonstrated live on a small synthetic logistic
model: SHAP contributions plus the base value must reconstruct the
model's own logit *exactly*, the same check `tests/test_explain.py` runs.

In [11]:
from sklearn.linear_model import LogisticRegression
from explainability.explain import linear_shap_values

rng = np.random.default_rng(1)
n, d = 500, 5
X_demo = pd.DataFrame(rng.normal(size=(n, d)), columns=[f"f{i}" for i in range(d)])
y_demo2 = (rng.uniform(size=n) < 0.3).astype(int)
model_demo = LogisticRegression().fit(X_demo, y_demo2)

shap_values, base_value = linear_shap_values(model_demo, X_demo, X_demo)
logit = model_demo.decision_function(X_demo)
reconstructed = shap_values.sum(axis=1) + base_value
print(f"max |reconstructed - actual logit| = {np.abs(reconstructed - logit).max():.2e}  (exact, not approximate)")

max |reconstructed - actual logit| = 2.22e-16  (exact, not approximate)


Now the real, headline finding on the actual data: `addr_state` ranks in
the SHAP top 10 of 96 features in both models, every fold, while its
permutation importance sits at essentially zero.

In [12]:
imp = pd.read_csv(REPORTS / "explainability_importance.csv")
addr_state = imp[imp.feature == "addr_state"]
addr_state[["fold", "model", "shap_importance", "perm_importance", "shap_rank", "perm_rank"]]

,fold,model,shap_importance,perm_importance,shap_rank,perm_rank
0,2014,logistic_woe,0.153643,-0.002203,3.0,96.0
101,2014,lightgbm,0.303625,0.003782,2.0,8.0
274,2015,logistic_woe,0.119635,0.001272,8.0,12.0
371,2015,lightgbm,0.233868,0.003382,3.0,7.0
385,2016,logistic_woe,0.087967,-0.002078,10.0,95.0
480,2016,lightgbm,0.127673,-0.001698,5.0,94.0
656,2017,logistic_woe,0.061917,0.000744,18.0,23.0
695,2017,lightgbm,0.101685,0.000399,6.0,32.0


Two explanations (small-sample overfitting, correlation with another
feature) were checked and ruled out — see finding 14 in the README and
decision 13 in DECISIONS.md. It's left there as an open item, not resolved
here, because `addr_state` is also where Phase 6's geographic fairness gap
comes from.

## 9. What this notebook doesn't show

Every number above came from a report file this project's own committed
code produced — `python run_all.py` regenerates every one of them from
raw data. This notebook's job was to narrate that work, not repeat it, and
its two live demonstrations (the threshold search, the exact SHAP closed
form) are the same mechanisms, not simplified stand-ins.

What's genuinely not covered here: the full leakage/imbalance/calibration
*mechanism* investigations (the SMOTE integer-rounding tell in finding 9,
the WoE encoder bug in finding 10, the two SHAP-library gotchas in Phase
7) — those are read, not re-run, in [`README.md`](../README.md) and
[`DECISIONS.md`](../DECISIONS.md). [`MODEL_CARD.md`](../MODEL_CARD.md)
has the deployment-relevant summary and caveats in one place.